# Retrieve and apply a baseline to a MagPy DataStream

The main workflow stays in MagPy: load a `DataStream`, retrieve an INTERMAGNET
HDZ reference, apply it to the stream, and inspect or save the corrected stream.
A short optional section at the end covers the legacy Valentia text scripts.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from magpy.stream import read

from magie import download_iaga2002
from magie.baseline import get_intermagnet_baseline, apply_intermagnet_baseline

## 1. Load a MagPy DataStream

Download one complete UTC day of Valentia minute IAGA-2002 data. For a local IAGA file,
replace the download call with `stream = read("/path/to/val20261028pmin.min")`.

In [2]:
site = "VAL"
observation_day = pd.Timestamp("2026-10-28", tz="UTC")
start = observation_day.normalize()
end = start + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

stream = download_iaga2002(
    start, end, site.lower(), intervals=("min",), save=False, n_jobs=1,
)
print(f"{len(stream['time'])} samples; components: {stream.header['DataComponents']}")

Downloading: 100.00% [106496/103234 bytes]
Download complete!
1440 samples; components: XYZF


## 2. Retrieve the HDZ reference

The reference date selects the **seven UTC days 16 through 10 days before it**,
inclusive. Here, 28 October 2026 selects 12–18 October. The result is a dictionary
with H and Z in nT and D in degrees, calculated as medians over usable reported
minute samples in that window. Missing values are excluded, and every requested
day must contain usable samples for each component.

The reference date is independent of the input period. Choose it explicitly for
repeatable processing; for a current reference use `pd.Timestamp.now(tz="UTC")`. The ten-day lag allows for publication delays
but does not guarantee availability; HTTP failures or insufficient data raise
errors.

In [3]:
reference_date = observation_day
baseline = get_intermagnet_baseline(reference_date, site)
baseline

{'H': 19653.19, 'D': -2.49, 'Z': 45006.37}

## 3. Apply the reference directly to the stream

Each UTC day and HDZ component uses `value - finite_daily_mean + baseline`.
XYZ input is converted to HDZ first. The same reference is used for every day
in a multi-day stream. Changing the input sample window changes its daily
means, so use a consistent window when comparing runs.

The function returns an independent XYZ stream in nT. It preserves the source,
timestamps, and gaps; derived F is recalculated from full-precision XYZ and an
independent scalar S is preserved. No file or network access occurs during
application. Adjusting the reference does not make the data definitive.

In [4]:
corrected = apply_intermagnet_baseline(stream, baseline)

print(corrected.header['DataComponents'])
display(pd.DataFrame({
    "UTC": pd.to_datetime(corrected["time"], utc=True),
    "X [nT]": np.asarray(corrected["x"], dtype=float),
    "Y [nT]": np.asarray(corrected["y"], dtype=float),
    "Z [nT]": np.asarray(corrected["z"], dtype=float),
}).head())

# Inspect daily HDZ means using a copy; corrected remains an XYZ stream.
hdz = corrected.copy().xyz2hdz()
hdz_values = pd.DataFrame(
    {component: np.asarray(hdz[key], dtype=float)
     for component, key in zip(("H", "D", "Z"), ("x", "y", "z"))},
    index=pd.to_datetime(hdz["time"], utc=True),
)
display(hdz_values.groupby(hdz_values.index.normalize()).mean())
display(pd.Series(baseline, name="Reference (H/Z: nT; D: deg)"))

XYZF


,UTC,X [nT],Y [nT],Z [nT]
0,2026-10-28 00:00:00+00:00,19638.355789,-848.739173,45009.820486
1,2026-10-28 00:01:00+00:00,19638.202902,-848.664634,45009.780486
2,2026-10-28 00:02:00+00:00,19638.127541,-849.059038,45009.510486
3,2026-10-28 00:03:00+00:00,19638.188620,-849.086848,45009.500486
4,2026-10-28 00:04:00+00:00,19638.485049,-848.985283,45009.630486


,H,D,Z
2026-10-28 00:00:00+00:00,19653.19,-2.49,45006.37


H    19653.19
D       -2.49
Z    45006.37
Name: Reference (H/Z: nT; D: deg), dtype: float64

## 4. Save the corrected DataStream as IAGA-2002

Continue using `corrected` with other MagPy/MagIE functions, or write daily IAGA
files directly with MagPy. This example writes provisional files under a local
output directory; rerunning may replace files with the same names. The XYZ
stream retains its full precision in memory; the IAGA writer formats the output.

In [5]:
output_dir = Path("baseline_tutorial_output")
output_dir.mkdir(parents=True, exist_ok=True)
corrected.write(
    str(output_dir / "iaga"),
    format_type="IAGA",
    coverage="day",
    filenamebegins=site.lower(),
    filenameends="pmin.min",  # use psec.sec for second-cadence input
    useg=False,
)

'baseline_tutorial_output/iaga/val20261028pmin.min'

## Optional: the legacy Valentia text route

The operational scripts split retrieval and application into separate jobs.
Save the same reference to JSON, then load it in the application job:

In [ ]:
import json

baseline_path = Path("baseline_tutorial_output/baseline.json")
with baseline_path.open("w", encoding="utf-8") as destination:
    json.dump(baseline, destination, indent=2)

# In the application job:
with baseline_path.open(encoding="utf-8") as source:
    baseline = json.load(source)
```

For the original hour/minute/HDZ Valentia feed, use the reader below instead of
the IAGA loader, apply the reference to its returned stream, and export legacy
XYZ text:

```python
from magie.baseline import read_val_realtime
from magie.file_conversions import write_magie_legacy_datastream

raw = read_val_realtime(
    "/path/to/original/val_mag_realtime.txt",
    "2026-10-28",  # actual UTC observation day: the feed contains no date
    declination_unit="legacy",
)
corrected = apply_intermagnet_baseline(raw, baseline)
write_magie_legacy_datastream(
    corrected, "baseline_tutorial_output/val_mag_realtime.txt"
)


`"legacy"` divides raw D by **360**, preserving the old script's arithmetic.
For known physical units choose `"deg"`, `"arcmin"`, `"arcsec"`, or `"nT"`;
`"nT"` represents transverse E and application uses
`D_degrees = degrees(E_nT / baseline_H_nT)`. `None` opts into the feed header's
unit. Select the interpretation that matches your source.

The legacy writer accepts one UTC day, writes XYZ to two decimals, omits F, and
uses `99999.99` for gaps. Keep the raw HDZ input and corrected XYZ output at
separate paths. The operational scripts use
`/mnt/data.magie.ie/mag_data/baseline.json` for the shared reference and
`/mnt/data.magie.ie/magnetometer_live/val_mag_realtime.txt` for corrected output.
The three-value JSON does not record site or reference date; retain that
provenance alongside it.